In [ ]:
import os
from typing import Any, Dict, List, TypedDict
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel,Field

from dotenv import load_dotenv
load_dotenv()

model = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
class UserContext(BaseModel):
    user_id: str = Field(..., description="The user ID of the user")


In [ ]:
USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson", 
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com",
        "support_tier": "Priority"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard", 
        "balance": 1200,
        "email": "bob@example.com",
        "support_tier": "Standard"
    }
}

In [ ]:
from langchain.tools import ToolRuntime

@tool
def get_account_details(runtime: ToolRuntime) :
    """ 
    Get account details 
    """

    user_id = runtime.context.user_id
    
    return USER_DATABASE[user_id]




In [ ]:
agent = create_agent(
    model=model,
    tools=[get_account_details],
    checkpointer=InMemorySaver(),
    context_schema=UserContext
    )

In [ ]:
result =agent.invoke(
    {"messages": [{"role": "user", "content": "What is my account balance?"}]},
    config={"configurable": {"thread_id": "123"}},
    context= UserContext(user_id="user123")
    )

result

In [ ]:
result['messages'][1].tool_calls

In [32]:
@tool
def save_ltm(data:str, runtime: ToolRuntime) :
    """ 
    Saves  data to longterm memory. use this tool when user shares personal information about themselves, 
    their health, likes, dislikes, interests, etc.
    """

    user_id = runtime.context.user_id
    store=runtime.store
    item= store.get(("users",user_id), "messages")
    if item is None:
        messages=[]
    else:
        messages=list(item.value)
   
    messages.append(data)

    store.put(("users",user_id), "messages", messages)

    return "Data saved to longterm memory"

@tool
def get_from_ltm(runtime: ToolRuntime) :
    """ 
    Retrieves data from longterm memory. use this  when prior facts are needed to be recalled.
    """
    user_id = runtime.context.user_id
    store=runtime.store
    item= store.get(("users",user_id), "messages")
    return list(item.value)
   


In [39]:
from langgraph.store.memory import InMemoryStore

agent = create_agent(
    model=model,
    tools=[save_ltm, get_from_ltm],
    checkpointer=InMemorySaver(),
    store=InMemoryStore(),
    context_schema=UserContext,
     system_prompt="""You are a helpful assistant that can answer any queries.
If the user shares personal information, preferences, or explicit approvals, heatlth information,
save that information using the save_ltm tool.
Before answering a query when prior facts might matter, retrieve long-term
memory using get_from_ltm and use that context in your answer.
""",
    )

In [34]:
result =agent.invoke(
    {"messages": [{"role": "user", "content": " I love to eat vada pav in mumbai"}]},
    config={"configurable": {"thread_id": "1"}},
    context= UserContext(user_id="user123")
    )

result

c:\npci-mumbai\ws\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user123'), input_type=UserContext])
  function=lambda v, h: h(v), schema=original_schema
c:\npci-mumbai\ws\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user123'), input_type=UserContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content=' I love to eat vada pav in mumbai', additional_kwargs={}, response_metadata={}, id='6e575d49-3e7b-466a-a386-9aa503584078'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 174, 'total_tokens': 198, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_5b9942f5fa', 'id': 'chatcmpl-ETOOAHRDqPiZv0FQRkgCmpK4ze7ME', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ec7c-56ed-7d92-b226-fd76d893fc2e-0', tool_calls=[{'name': 'save_ltm', 'args': {'data': 'User loves to eat vada pav in Mumbai.'}, 

In [35]:
result['messages'][1].tool_calls

[{'name': 'save_ltm',
  'args': {'data': 'User loves to eat vada pav in Mumbai.'},
  'id': 'call_uw6wUSV5V6fL5Lo5ySKu4los',
  'type': 'tool_call'}]

In [ ]:
result =agent.invoke(
    {"messages": [{"role": "user", "content": "What will i prefer to eat in mumbai"}]},
    config={"configurable": {"thread_id": "1"}},
    context= UserContext(user_id="user123")
    )

result

c:\npci-mumbai\ws\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user124'), input_type=UserContext])
  function=lambda v, h: h(v), schema=original_schema
c:\npci-mumbai\ws\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user124'), input_type=UserContext])
  return self.__pydantic_serializer__.to_python(


AttributeError: 'NoneType' object has no attribute 'value'

In [37]:
result['messages'][1].tool_calls

[{'name': 'get_from_ltm',
  'args': {},
  'id': 'call_x5rItXWqi6NbSwycysXXA5Yu',
  'type': 'tool_call'}]